# M0 · Un cono, dos respuestas: ON y OFF

**Pregunta central:** ¿cómo una misma reducción de glutamato puede hacer subir el voltaje de una bipolar y bajar el de otra?

Este laboratorio recorre **fenómeno → variables → balances → receptores → predicción → simulación → interpretación**. Son tres células distintas, cada una con un compartimento; no son los tres compartimentos O–I–T del bastón. Las curvas son **simulaciones con parámetros ilustrativos y no calibrados**. No hay ganglionares, spikes, GABA, adaptación ni una red espacial.

Necesitas EDO y nociones de corriente; la fisiología se introduce aquí. Lee las preguntas antes de ejecutar. Las celdas llaman a los módulos M0: no contienen una segunda implementación del modelo.

[Derivación y unidades](../docs/modelo_cono_on_off_minimo.md) · [Fuentes](../docs/referencias_cono_on_off.md) · [Preguntas razonadas](../docs/preguntas_cono_on_off.md)

**Referencia conceptual principal:** Sterling y Demb, *Retina* (2004), fig. 6.15, p. 247, y receptores ON/OFF, p. 251. Se consultó una copia pública. Nuestros esquemas son originales y representan solo el tramo cono→bipolares.

## 00 · Preparación

Ejecuta en orden con el kernel del repositorio. `ipywidgets` es opcional: cada explorador tiene una función Python equivalente. Los productos se guardan en `outputs/retina_on_off/`; conserva este notebook fuente sin salidas al versionarlo.

In [ ]:
from pathlib import Path
from dataclasses import replace, asdict
import numpy as np
# Registrar imágenes embebidas también al ejecutar con MPLBACKEND=Agg (CI).
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from neurofisiologia.retina_on_off import (
    M0Parameters, M0Result, LightStep, LightProtocol, background_equilibrium,
    constant_light_equilibrium, release_fraction, synaptic_conductances,
    current_components, rhs, simulate_m0,
)
from neurofisiologia.retina_on_off_plotting import (
    PARAMETER_LABELS, pulse_protocol, paired_light_experiment, parameter_comparison,
    plot_physiology, plot_electrical, plot_cone, plot_transfer, plot_causal_chain,
    plot_voltage_experiments, save_figure,
)
ROOT = Path.cwd() if (Path.cwd()/"src").is_dir() else Path.cwd().parent
OUTPUT = ROOT/"outputs"/"retina_on_off"
p = M0Parameters()
times = np.linspace(0, 600, 601)

def show_figure(figure: plt.Figure, name: str) -> None:
    """Muestra y exporta una figura simulada; cierra el objeto para liberar memoria."""
    save_figure(figure, name, OUTPUT)
    display(figure)
    plt.close(figure)


## 01 · Antes de las ecuaciones: ¿qué cambia con la luz?

Un potencial de membrana es la diferencia interior–exterior. **Despolarizar** significa subir (por ejemplo de −48 a −44 mV); **hiperpolarizar**, bajar. No hace falta cruzar cero. Una respuesta **graduada** puede ser pequeña o grande sin disparar un potencial de acción.

Trabajaremos alrededor de **L0=1**, no alrededor de oscuridad. Un incremento lleva L de 1 a 1.5; un decremento lo lleva de 1 a 0.5. Ambas iluminaciones son no negativas. ON y OFF se refieren a respuestas a esos cambios relativos, no a que una lámpara esté meramente encendida o apagada.

El cono libera una señal química hacia las dos bipolares. En el esquema siguiente las flechas discontinuas transmiten información sobre glutamato; **no representan corriente axial**.

In [ ]:
show_figure(plot_physiology(), "M0_01_fisiologia")

## 02 · Construir el cono desde conservación de carga

La corriente entrante neta carga la membrana: $C_C\dot V_C=I_{neta}$. La fuga $-g_{L,C}(V_C-E_C)$ tiende a restaurar $E_C$. Representamos el efecto de la luz por una **variación** efectiva de corriente:

$$I_{foto}(t)=-k_L[L(t)-L_0],\qquad
C_C\dot V_C=-g_{L,C}(V_C-E_C)+I_{foto}(t).$$

La luz no es carga eléctrica. `I_foto=0` en el fondo significa variación nula, no ausencia de corriente basal real. Esta aproximación no incluye adaptación ni fotosensibilidad espectral.

**Pregunta:** ¿qué signo tiene la derivada justo después de aumentar L desde el fondo?  
**Predicción:** I_foto se hace negativa; V_C empieza a bajar. Con menos luz ocurre lo contrario.  
**Experimento B/C, primera parte:** cambia únicamente el signo de ΔL. Todo lo demás queda fijo.

In [ ]:
increment, decrement = paired_light_experiment(0.5)
show_figure(plot_cone([increment, decrement]), "M0_03_cono")

**Interpretación:** L y la corriente cambian por pasos; V_C es continuo porque la membrana tiene capacitancia. Para L sostenida,
$V_C^\infty=E_C-k_L(L-L_0)/g_{L,C}$ y $\tau_C=C_C/g_{L,C}$.
La malla muestra una muestra por ms, pero el integrador separa exactamente todos los saltos. Un pulso entre muestras también se integra.

| Magnitud | Unidad | Comprobación |
|---|---|---|
| t, V | ms, mV | V es interior menos exterior |
| C total | pF | pF·mV/ms = pA |
| g total | nS | nS·mV = pA |
| I entrante | pA | Positiva aumenta V |
| L, r | Adimensionales | No son irradiancia ni concentración calibradas |

## 03 · Del voltaje del cono al glutamato

No añadiremos una EDO química. Adoptamos una salida instantánea:

$$r(V_C)=\frac{1}{1+\exp[-(V_C-V_{1/2})/s]},\qquad s>0.$$

r representa una fracción de liberación, no una concentración medida. La forma sigmoidal y sus constantes son una **hipótesis de M0**. Como $dr/dV_C=r(1-r)/s>0$, la despolarización aumenta r. En V_half, r=1/2. La función numérica evita overflow; en extremos puede redondear a 0 o 1.

**Pregunta:** si la luz hiperpolariza el cono, ¿qué hace r?  
**Predicción:** disminuye.  
**Experimento:** recorre de izquierda a derecha la curva de liberación y después ambas curvas de apertura.

In [ ]:
show_figure(plot_transfer(p), "M0_04_transferencia")

## 04 · Primero OFF: menos apertura catiónica puede hiperpolarizar

Los receptores ionotrópicos OFF se representan por
$g_{OFF}=g_{OFF,max}r$. La corriente sináptica es
$g_{OFF}(E_{syn,OFF}-V_{OFF})$:

$$C_{OFF}\dot V_{OFF}=-g_{L,OFF}(V_{OFF}-E_{L,OFF})
+g_{OFF}(r)(E_{syn,OFF}-V_{OFF}).$$

Si E_syn es mayor que V_OFF, esa corriente es entrante. Con menos glutamato se reduce la apertura y disminuye la entrada; la fuga puede hacer bajar V_OFF. **No es necesario añadir una corriente inhibitoria** para reducir un potencial.

## 05 · Después ON: invertir la apertura, no el voltaje

mGluR6 controla indirectamente canales catiónicos. M0 resume esa acción como
$g_{ON}=g_{ON,max}(1-r)$:

$$C_{ON}\dot V_{ON}=-g_{L,ON}(V_{ON}-E_{L,ON})
+g_{ON}(r)(E_{syn,ON}-V_{ON}).$$

Las dos EDO bipolares tienen la misma forma. En ON, menos glutamato aumenta g y permite más entrada. Nunca usamos conductancia negativa, reversión de Cl⁻ ni un cambio artificial de signo del voltaje. La asociación molecular con TRPM1 procede de evidencia posterior al capítulo de 2004; aquí no simulamos la cascada.

**Predicción antes del esquema:** cada célula debe tener una C y una fuga propias; solo las bipolares necesitan una rama de conductancia sináptica. Ningún cable une los nodos de células distintas.

In [ ]:
show_figure(plot_electrical(p), "M0_02_circuitos")

## 06 · Experimento A: encontrar el verdadero reposo

**Pregunta:** con sinapsis basal abierta, ¿puede V_ON empezar en su reversión de fuga?  
**Predicción:** no, porque allí la corriente sináptica no está compensada.

En L=L0: $V_C^*=E_C$, $r^*=r(E_C)$. Para cada bipolar b:

$$V_b^*=\frac{g_{L,b}E_{L,b}+g_b(r^*)E_{syn,b}}{g_{L,b}+g_b(r^*)}.$$

Es un promedio ponderado por conductancias. **Experimento:** mantén L constante, calcula el equilibrio y verifica que las tres derivadas se anulan. Ninguna variable de control cambia.

In [ ]:
equilibrium = background_equilibrium(p)
background_result = simulate_m0(times, p, LightProtocol())
print("V* (cono, ON, OFF), mV:", equilibrium)
print("r*:", float(release_fraction(equilibrium[0], p)))
print("g* (ON, OFF), nS:", synaptic_conductances(release_fraction(equilibrium[0], p), p))
print("RHS de fondo, mV/ms:", rhs(0, equilibrium, p, LightProtocol()))
assert np.max(np.abs(background_result.delta_voltage_mv)) < 1e-9
show_figure(plot_voltage_experiments([background_result], ["A · Fondo constante L0=1"]), "M0_05_fondo")

**Interpretación:** con nuestros parámetros, V*=(-40,-48,-48) mV. Las bipolares no están en −60 mV, su E_L. Si cambiaras fuga o gmax, habría que recalcular su reposo antes de comparar. Todos estos valores son ilustrativos y no pertenecen a una especie calibrada.

## 07 · B/C completos: seguir toda la cadena causal

**Pregunta:** ¿dónde aparece la inversión ON?  
**Predicción:** antes del voltaje bipolar, en la dependencia de g respecto a r.

| Cambio respecto a L0 | V_C | r | g_ON | g_OFF | V_ON | V_OFF |
|---|---|---|---|---|---|---|
| Más luz | ↓ | ↓ | ↑ | ↓ | ↑ | ↓ |
| Menos luz | ↑ | ↑ | ↓ | ↑ | ↓ | ↑ |

**Experimento:** compara columnas con idénticas escalas y parámetros. El control cambia únicamente la magnitud de ΔL; se muestran simultáneamente sus dos signos. Cero produce fondo constante; uno permite alcanzar L=0 en el decremento. En la alternativa Python llama `compare_light(0.25)` o `compare_light(1.0)`.

In [ ]:
def compare_light(contrast: float = 0.5) -> tuple[M0Result, M0Result]:
    """Compara signos de un mismo contraste alrededor de L0=1, sin cambiar membranas."""
    global light_comparison
    light_comparison = paired_light_experiment(contrast)
    figure = plot_causal_chain(light_comparison,
                              [f"B · Incremento: 1 → {1+contrast:g}", f"C · Decremento: 1 → {1-contrast:g}"])
    display(figure)
    plt.close(figure)
    return light_comparison

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is not None:
    light_controls = widgets.interactive(compare_light,
        contrast=widgets.FloatSlider(value=.5, min=0, max=1, step=.1,
                                     description="|ΔL|", continuous_update=False))
    display(light_controls)
    light_controls.update()
else:
    print("Sin widgets: usa compare_light(contrast=0.5).")
    compare_light(.5)
show_figure(plot_causal_chain([increment, decrement], ["B · Más luz", "C · Menos luz"]), "M0_06_cadena")

**Interpretación:** r y g son salidas algebraicas; siguen al voltaje del cono. Las bipolares integran corrientes, por lo que no siguen instantáneamente la apertura. Las flechas de la tabla no predicen igualdad de amplitudes ni simetría perfecta.

**Por qué los signos emergen:** para una conductancia constante,
$$\frac{dV_b^\infty}{dg}=\frac{g_{L,b}(E_{syn,b}-E_{L,b})}{(g_{L,b}+g)^2}>0.$$
Junto con $dV_C^\infty/dL<0$, $dr/dV_C>0$ y las pendientes opuestas de g_ON/g_OFF, esto determina los signos. No hay un término +L o −L añadido directamente a las bipolares.

## 08 · Mirar las corrientes, no solo las curvas

**Pregunta:** mientras una bipolar sube y otra baja, ¿deben las dos corrientes sinápticas tener signos contrarios?  
**Predicción:** no. Ambas pueden ser entrantes, pero de diferente magnitud respecto a la fuga.  
**Experimento:** fija el instante en 120 ms después del incremento y suma corrientes por célula.

In [ ]:
sample = int(np.argmin(abs(increment.time_ms-120)))
components = current_components(increment.voltage_mv[:, sample], increment.illumination[sample],
                                increment.protocol.background, p)
rows = ["| Célula | Fuga (pA) | Foto (pA) | Sináptica (pA) | Neta (pA) | dV/dt (mV/ms) |",
        "|---|---:|---:|---:|---:|---:|"]
for i, name in enumerate(("Cono", "ON", "OFF")):
    rows.append(f"| {name} | {components['leak_pa'][i]:.4f} | {components['photo_pa'][i]:.4f} | "
                f"{components['synaptic_pa'][i]:.4f} | {components['net_pa'][i]:.4f} | "
                f"{components['net_pa'][i]/p.capacitance_pf[i]:.4f} |")
display(Markdown("**Simulación en t=120 ms; corriente entrante positiva.**\n\n"+"\n".join(rows)))

**Interpretación:** divide la suma neta, no cada corriente con un signo inventado, por la C de la célula. La bipolar OFF puede tener corriente sináptica positiva y corriente neta negativa. No se restan las corrientes bipolares del cono: son corrientes locales activadas por una señal química, sin balance axial entre células.

## 09 · Experimento D: regreso al fondo y pulsos sucesivos

**Pregunta:** ¿vuelven todos los voltajes al reposo en el mismo instante que L?  
**Predicción:** la corriente fotosensible cambia de inmediato, pero la carga de membrana tarda en relajarse.  
**Experimento:** cambia solo la secuencia temporal de L, conserva los parámetros y deja un tramo final suficientemente largo de fondo.

In [ ]:
train = LightProtocol(steps=(LightStep(100,200,1.5), LightStep(300,400,.5), LightStep(500,600,1.5)))
train_result = simulate_m0(np.linspace(0,1100,1101), p, train)
show_figure(plot_causal_chain([train_result], ["D · Incremento, decremento y segundo incremento"]), "M0_07_pulsos")
print("Desviación final respecto a reposo (mV):", train_result.delta_voltage_mv[:, -1])
short_result = simulate_m0(np.linspace(0,400,801), p, LightProtocol(steps=(LightStep(100,150,1.5),)))
print("Instante de mínimo del cono (ms):", short_result.time_ms[np.argmin(short_result.voltage_mv[0])])
print("Instante de máximo ON (ms):", short_result.time_ms[np.argmax(short_result.voltage_mv[1])])

**Interpretación:** un máximo bipolar puede ocurrir después del extremo del cono. No mide una latencia vesicular: M0 solo tiene tres variables de voltaje y liberación algebraica. Para una conductancia fija, $\tau_b=C_b/(g_{L,b}+g_b)$; durante el estímulo g_b cambia, así que esa escala es local y variable.

## 10 · Experimento E: desconectar una rama

**Pregunta:** ¿qué deja de responder cuando g_ON,max=0?  
**Predicción:** únicamente ON; el cono y OFF siguen recibiendo y procesando la misma entrada.  
**Experimento:** cada columna modifica un máximo sináptico y recalcula el reposo correspondiente. La intervención OFF es independiente de la ON.

In [ ]:
isolated_on = simulate_m0(times, replace(p, g_on_max_ns=0), pulse_protocol())
isolated_off = simulate_m0(times, replace(p, g_off_max_ns=0), pulse_protocol())
show_figure(plot_voltage_experiments([increment, isolated_on, isolated_off],
                                    ["Referencia", "Sin sinapsis ON", "Sin sinapsis OFF"]), "M0_08_aislamiento")

**Interpretación:** la rama desconectada queda en su E_L; su ΔV es cero. El cambio de reposo absoluto no es una respuesta a la luz. Que no cambien las otras dos trayectorias pone a prueba la ausencia de realimentación.

## 11 · Experimento F: qué cambia C, fuga o ganancia

**Pregunta:** ¿cambiar rapidez equivale a cambiar el equilibrio?  
**Predicción:** C no aparece en la ecuación del equilibrio, pero sí en el tiempo. En un pulso finito, cambiar rapidez puede cambiar el máximo observado. Fugas y gmax sí pueden cambiar el reposo; k_L cambia la ganancia de luz.

**Experimento:** selecciona **un** parámetro y un factor. El estímulo queda fijo (L0=1, ΔL=0.5, 100–300 ms). La referencia discontinua es siempre la misma. La línea continua muestra el modificado. Los ΔV usan los reposos respectivos. `compare_parameter("C_ON", 2)` funciona también sin widgets.

Prueba C de cada célula, k_L, fugas y gmax. Antes de cada cambio escribe qué curvas deberían permanecer idénticas.

In [ ]:
def compare_parameter(parameter: str = "C_ON", factor: float = 1.0) -> tuple[M0Result, M0Result]:
    """Modifica un solo parámetro; mantiene una referencia canónica independiente."""
    global parameter_results
    parameter_results = parameter_comparison(parameter, factor)
    reference, modified = parameter_results
    print(f"{PARAMETER_LABELS[parameter]}: factor {factor:g}; iluminación idéntica.")
    print("C (pF):", modified.parameters.capacitance_pf, "Fugas (nS):", modified.parameters.leak_ns)
    print("k_L (pA/u. L):", modified.parameters.light_gain_pa,
          "gmax ON/OFF (nS):", modified.parameters.g_on_max_ns, modified.parameters.g_off_max_ns)
    print("Reposo referencia (mV):", reference.equilibrium_mv, "Modificado:", modified.equilibrium_mv)
    figure = plot_causal_chain([modified], [f"F · {PARAMETER_LABELS[parameter]} × {factor:g}"], references=[reference])
    display(figure)
    plt.close(figure)
    return parameter_results

if widgets is not None:
    parameter_controls = widgets.interactive(compare_parameter,
        parameter=widgets.Dropdown(options=[(label,key) for key,label in PARAMETER_LABELS.items()], value="C_ON", description="Parámetro", layout=widgets.Layout(width="350px")),
        factor=widgets.FloatSlider(value=1, min=.25, max=2, step=.25, description="Factor", continuous_update=False))
    display(parameter_controls)
    parameter_controls.update()
else:
    compare_parameter("C_ON", 1)
reference_c, modified_c = parameter_comparison("C_ON", 2)
show_figure(plot_causal_chain([modified_c], ["F · Doble C_ON; entrada y demás parámetros fijos"], references=[reference_c]), "M0_09_sensibilidad")

**Interpretación:** aumentar C_ON retrasa ON sin alterar cono ni OFF. Cambiar la fuga del cono afecta las dos ramas porque cambia la señal compartida; cambiar una fuga bipolar no afecta la otra rama. Cerca de los extremos de r hay saturación de liberación en esta hipótesis, aunque luz→corriente siga siendo lineal.

## 12 · Qué hemos deducido y qué hemos supuesto

Para L constante, la Jacobiana es triangular: sus autovalores son
$-g_{L,C}/C_C$, $-(g_{L,ON}+g_{ON})/C_{ON}$ y $-(g_{L,OFF}+g_{OFF})/C_{OFF}$, todos negativos. El reposo es estable. Esto es una consecuencia del M0, no una demostración de que una retina real solo tenga estos mecanismos.

| Categoría | Ejemplo |
|---|---|
| Motivación fisiológica | Distinción de receptores y polaridad ON/OFF respecto al fondo |
| Hipótesis de M0 | Luz→corriente lineal, r sigmoidal instantánea, g_ON∝1−r y g_OFF∝r |
| Deducción | Reposo ponderado, signos y constantes de tiempo |
| Resultado numérico | Trayectorias, máximos y relajación de los experimentos |

No se importaron parámetros de bastón de sapo. La simetría elegida para ON/OFF permite comparar mecanismos, pero no afirma igualdad entre tipos celulares reales.

## 13 · Qué datos necesitaríamos para calibrar

Definir especie y tipo celular; medir iluminación absoluta y fondo/adaptación, corriente fotosensible y V_C, capacitancias y fugas, reversión postsináptica, liberación o un indicador validado y respuestas ON/OFF bajo el mismo protocolo. Añadir incertidumbre y comprobar predicciones fuera del ajuste. Una traza de voltaje no identifica por sí sola todas las conductancias.

**Límites:** no hay adaptación, fotosensibilidad espectral, Ca²⁺, cinética de vesículas, sinapsis de cinta detallada, cascada mGluR6/TRPM1 explícita, horizontales, amacrinas, GABA, ganglionares, spikes, cable o red. r no es concentración y L no es flujo fotónico calibrado. Estos límites no deben ocultarse al interpretar las figuras.

## 14 · Autoevaluación

1. ¿Puede OFF hiperpolarizar mientras su corriente sináptica sigue siendo entrante?
2. ¿Qué transitorio artificial aparece si inicializas ON en E_L con sinapsis activa?
3. ¿Por qué un pulso corto puede producir menor máximo al aumentar C, aunque el equilibrio sea igual?
4. ¿Qué observación distinguiría una inversión por receptor de una multiplicación artificial del voltaje por −1?

<details><summary>Respuestas razonadas</summary>

1. Sí: importa la suma de entrada y fuga; reducir una entrada puede hacer negativa la neta.
2. Una despolarización incluso con L constante, pues V no estaba en el equilibrio ponderado.
3. El pulso termina antes de aproximarse al equilibrio; compara tiempos y no solo máximos.
4. Comprobar g_ON/g_OFF no negativas, fuerzas impulsoras catiónicas, corrientes, ausencia de respuesta bipolar instantánea al salto de L y desconexión selectiva. Con parámetros asimétricos las amplitudes no deben quedar artificialmente espejadas.

</details>

Consulta las [diez preguntas ampliadas](../docs/preguntas_cono_on_off.md) y la
[evidencia de validación](../docs/validacion_cono_on_off.md).